In [1]:
import sys
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

root = str(Path.cwd().parent)

if root not in sys.path:
    sys.path.append(root)

from src.step2_transformation import DataTransformer
from src.step3a_bwm_model import BWMCalculator
from src.step4_saw_aggregation import SAWCalculator

In [2]:
# =========================================================
# 1. PHASE 2: NORMALIZATION (The Data Engineering)
# =========================================================
transformer = DataTransformer(config_name='step1_features_config.yaml', max_budget=105000)
df_normalized = transformer.execute_pipeline()

✅ YAML Config loaded successfully.

🚀 Starting ETL pipeline for file: data/car_database.csv
Filter 1: Applying Hard Constraints dynamically...
  -> Applied constraint: cost <= 105000
 -> Cars removed: 17 | Remaining: 14
Filter 2: Removing uninformative columns...
  -> Auto-dropped (Zero Variance): ['hill_start_assist', 'abs', 'driver_seat_height_adjustment', 'leather_seats', 'transmission', 'traction_control', 'power_steering', 'handling', 'stability_control', 'leather_steering_wheel', 'power_door_locks']
Transformation 1: Applying Conditional Logic dynamically...
  -> Applied FILL_NULL (1.0) for: fog_lights
  -> Applied OR_GATE substitution for: rear_view_camera
  -> Applied OR_GATE substitution for: rear_parking_sensors
Transformation 2: Applying Utility Mapping...
Transformation 3: Binarizing booleans...
Transformation 4: Continuous Normalization (Min-Max)...
✅ Pipeline Finished! Mathematical matrix generated successfully.



In [3]:
# =========================================================
# BWM TEMPLATE GENERATOR
# =========================================================

# 1. Get all columns from df_normalized, ignoring the IDs
id_columns = ['car', 'version'] # If you have more IDs in your YAML, add them here
real_criteria = [col for col in df_normalized.columns if col not in id_columns]

print(f"✅ Found {len(real_criteria)} valid criteria with variance:\n")
for i, crit in enumerate(real_criteria):
    print(f"  {i+1}. {crit}")

print("\n" + "="*50)
print("COPY AND PASTE THE CODE BELOW INTO A NEW CELL AND FILL IN THE SCORES (1-9):")
print("="*50 + "\n")

# 2. Generate the Python code skeleton for you to copy and fill
template_code = f"""criteria_list = {real_criteria}

# CHOOSE YOUR FAVORITE AND YOUR LEAST FAVORITE (Copy the exact name from the list above)
best_criterion = "PUT_THE_NAME_HERE"
worst_criterion = "PUT_THE_NAME_HERE"

# SCORES FROM 1 TO 9 (How much more important is the 'best_criterion' than the criterion below?)
# Delete the line of the criterion you chose as the 'best'
bo_vector = {{"""

for crit in real_criteria:
    template_code += f"\n    '{crit}': 1, # Adjust this score"
template_code += "\n}\n"

template_code += """
# SCORES FROM 1 TO 9 (How much more important is the criterion below than the 'worst_criterion'?)
# Delete the line of the criterion you chose as the 'worst'
ow_vector = {"""

for crit in real_criteria:
    template_code += f"\n    '{crit}': 1, # Adjust this score"
template_code += "\n}"

print(template_code)

✅ Found 28 valid criteria with variance:

  1. cost
  2. warranty
  3. engine
  4. horsepower
  5. torque
  6. cruise_control
  7. city_fuel_economy
  8. highway_fuel_economy
  9. ground_clearance
  10. payload_capacity
  11. airbag
  12. air_conditioning
  13. infotainment_system
  14. keyless
  15. steering_wheel_adjustment
  16. power_windows
  17. power_side_mirrors
  18. rear_view_camera
  19. rear_parking_sensors
  20. tire_pressure_sensor
  21. twilight_sensor
  22. fog_lights
  23. led_headlights
  24. roof_rails
  25. alloy_wheels
  26. tow_hitch
  27. aesthetics
  28. freshness

COPY AND PASTE THE CODE BELOW INTO A NEW CELL AND FILL IN THE SCORES (1-9):

criteria_list = ['cost', 'warranty', 'engine', 'horsepower', 'torque', 'cruise_control', 'city_fuel_economy', 'highway_fuel_economy', 'ground_clearance', 'payload_capacity', 'airbag', 'air_conditioning', 'infotainment_system', 'keyless', 'steering_wheel_adjustment', 'power_windows', 'power_side_mirrors', 'rear_view_camera', '

In [4]:
# =========================================================
# 2. PHASE 3: BWM WEIGHTS (The Mathematical Brain)
# =========================================================
# We extract the criteria dynamically from the normalized DataFrame
criteria_list = ['cost', 'warranty', 'engine', 'horsepower', 'torque', 'cruise_control', 'city_fuel_economy', 'highway_fuel_economy', 'ground_clearance', 'payload_capacity', 'airbag', 'air_conditioning', 'infotainment_system', 'keyless', 'steering_wheel_adjustment', 'power_windows', 'power_side_mirrors', 'rear_view_camera', 'rear_parking_sensors', 'tire_pressure_sensor', 'twilight_sensor', 'fog_lights', 'led_headlights', 'roof_rails', 'alloy_wheels', 'tow_hitch', 'aesthetics', 'freshness']

# CHOOSE YOUR FAVORITE AND YOUR LEAST FAVORITE
best_criterion = "cost"
worst_criterion = "keyless"

# VETOR BO (Best-to-Others): 
bo_vector = {
    'cruise_control': 5,
    'airbag': 2,                  
    'city_fuel_economy': 8,       
    'engine': 3,                 
    'air_conditioning': 7,       
    'warranty': 7,               
    'highway_fuel_economy': 8,    
    'horsepower': 5,              
    'torque': 5,                  
    'rear_view_camera': 5,        
    'power_windows': 6,           
    'power_side_mirrors': 8,      
    'infotainment_system': 3,     
    'rear_parking_sensors': 5,    
    'steering_wheel_adjustment': 6, 
    'ground_clearance': 4,        
    'payload_capacity': 7,        
    'fog_lights': 5,              
    'twilight_sensor': 8,         
    'tire_pressure_sensor': 8,    
    'led_headlights': 8,          
    'roof_rails': 6,              
    'alloy_wheels': 8,            
    'tow_hitch': 6,               
    'aesthetics': 8,              
    'freshness': 8,               
    'keyless': 9                 
}

# VETOR OW (Others-to-Worst): 
ow_vector = {
    'cruise_control': 5,
    'cost': 9,                    
    'airbag': 5,                  
    'city_fuel_economy': 4,       
    'engine': 6,                  
    'air_conditioning': 3,        
    'warranty': 3,                
    'highway_fuel_economy': 3,    
    'horsepower': 2,              
    'torque': 6,                  
    'rear_view_camera': 6,        
    'power_windows': 4,           
    'power_side_mirrors': 3,      
    'infotainment_system': 7,     
    'rear_parking_sensors': 6,    
    'steering_wheel_adjustment': 7, 
    'ground_clearance': 7,        
    'payload_capacity': 5,        
    'fog_lights': 6,              
    'twilight_sensor': 1,         
    'tire_pressure_sensor': 1,    
    'led_headlights': 1,          
    'roof_rails': 4,              
    'alloy_wheels': 1,            
    'tow_hitch': 4,               
    'aesthetics': 1,              
    'freshness': 1               
}

bwm = BWMCalculator(criteria_list)
weights, cr = bwm.calculate_weights(best_criterion, worst_criterion, bo_vector, ow_vector)

In [5]:
# =========================================================
# 3. PHASE 5: SAW AGGREGATION (The Final Decision)
# =========================================================
saw = SAWCalculator()
df_final_ranking = saw.calculate_ranking(df_normalized, weights)

Aggregation: Calculating Final SAW Scores...
✅ Ranking generated successfully!


In [6]:
# =========================================================
# 4. VISUALIZATION: The Top 5 Cars
# =========================================================
print(f"📊 BWM Consistency Ratio: {cr}")

# Select only the columns that matter for the final user report
columns_to_show = ['car', 'version', 'Final_Score'] + criteria_list
df_top5 = df_final_ranking[columns_to_show].head(5)

# Render a beautiful DataFrame highlighting the final score
display(df_top5)

📊 BWM Consistency Ratio: 0.0081


,car,version,Final_Score,cost,warranty,engine,horsepower,torque,cruise_control,city_fuel_economy,highway_fuel_economy,ground_clearance,payload_capacity,airbag,air_conditioning,infotainment_system,keyless,steering_wheel_adjustment,power_windows,power_side_mirrors,rear_view_camera,rear_parking_sensors,tire_pressure_sensor,twilight_sensor,fog_lights,led_headlights,roof_rails,alloy_wheels,tow_hitch,aesthetics,freshness
0,Peugeot 208 2026,1.0 FIREFLY FLEX STYLE MANUAL,0.633090,0.412121,0.0,0.0,0.000000,0.105263,1.0,1.000000,0.642857,0.8,0.266667,0.8,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.000000,1.00
1,Fiat Argo 2026,1.3 FIREFLY FLEX TREKKING MANUAL,0.620873,0.000000,0.0,1.0,1.000000,1.000000,0.0,0.000000,0.000000,1.0,0.000000,0.5,0.8,1.0,0.0,0.5,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.666667,0.50
2,Hyundai Hb20 2026,1.0 12V FLEX LIMITED MANUAL,0.608093,0.212121,1.0,0.0,0.133333,0.000000,1.0,0.800000,0.678571,0.8,0.000000,1.0,0.8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.333333,0.00
3,Citroën C3 2026,1.0 FIREFLY FLEX XTR MANUAL,0.586209,0.696970,0.0,0.0,0.000000,0.157895,0.0,0.600000,0.250000,1.0,1.000000,0.5,1.0,1.0,0.0,0.5,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.333333,0.75
4,Citroën C3 2026,1.0 FIREFLY FLEX FEEL MANUAL,0.575518,0.878788,0.0,0.0,0.000000,0.157895,0.0,0.733333,0.357143,1.0,1.000000,0.5,0.8,1.0,0.0,0.5,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.333333,0.75
